# Data Engineering - Multi-Station NDBC Data Processing
Processing wave data for stations 32012 and 51028

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# 📊 Create DataFrames for Multi-Station NDBC Data

# Define stations and their year ranges
stations_config = {
    '51028': list(range(2005, 2009)),   # 2005-2008 (post 2005)
    '41008': list(range(2006, 2010)) + list(range(2013, 2026))
}

# Map file types to actual filenames
file_type_mapping = {
    'met': 'stdmet',
    'den': 'swden', 
    'dir1': 'swdir',
    'dir2': 'swdir2',
    'r1': 'swr1',
    'r2': 'swr2'
}

file_types = ['met', 'den', 'dir1', 'dir2', 'r1', 'r2']

# Base data directory
base_data_dir = Path('../data/ndbc')

print(f"📋 Station Configuration:")
for station, years in stations_config.items():
    print(f"   Station {station}: {min(years)}-{max(years)} ({len(years)} years)")

print(f"\n📁 File types to process: {file_types}")
print(f"📂 File mapping: {file_type_mapping}")
print(f"📂 Base data directory: {base_data_dir}")

# Initialize counters
total_files = sum(len(years) * len(file_types) for years in stations_config.values())
files_processed = 0
files_missing = 0
dataframes_created = 0

print(f"\n🎯 Expected total files: {total_files}")
print(f"\n🚀 Starting dataframe creation...")

# Loop through stations, years, and file types
for station, years in stations_config.items():
    print(f"\n📍 Processing Station {station}...")
    
    for year in years:
        print(f"  📅 Year {year}:", end=" ")
        
        for file_type in file_types:
            # Construct dataframe name
            df_name = f"{file_type}_{station}_{year}"
            
            # Get actual filename
            actual_filename = file_type_mapping[file_type]
            
            # Construct file path: data/ndbc/station_number/year/station_number-year-filename.txt
            file_path = base_data_dir / station / str(year) / f"{station}-{year}-{actual_filename}.txt"
            
            if file_path.exists():
                try:
                    # Handle different reading logic for met files vs others
                    if file_type == 'met':
                        # For met files: keep header (row 0), skip units row (row 1)
                        df = pd.read_csv(file_path, sep='\s+', skiprows=[1])
                    else:
                        # For other files: keep header (row 0) only
                        df = pd.read_csv(file_path, sep='\s+')
                    
                    # Remove '#' from column names for all files
                    df.columns = df.columns.str.replace('#', '', regex=False)
                    
                    # Create the dataframe in global namespace
                    globals()[df_name] = df
                    
                    dataframes_created += 1
                    print(f"{file_type}✓", end=" ")
                    
                except Exception as e:
                    print(f"{file_type}✗({str(e)[:10]})", end=" ")
            else:
                print(f"{file_type}❓", end=" ")
                files_missing += 1
            
            files_processed += 1
        
        print()  # New line after each year

print(f"\n✅ Dataframe Creation Complete!")
print(f"   📊 Total dataframes created: {dataframes_created}")
print(f"   ❌ Files missing: {files_missing}")
print(f"   📁 Files processed: {files_processed}")
print(f"   📈 Success rate: {(dataframes_created/files_processed)*100:.1f}%")

# Show available dataframes
print(f"\n📋 Available DataFrames:")
available_dfs = [name for name in globals().keys() if any(f"{ft}_" in name for ft in file_types) and any(station in name for station in stations_config.keys())]
available_dfs.sort()

for station in stations_config.keys():
    station_dfs = [df for df in available_dfs if station in df]
    if station_dfs:
        print(f"  Station {station}: {len(station_dfs)} dataframes")
        # Group by file type
        for file_type in file_types:
            type_dfs = [df for df in station_dfs if df.startswith(file_type)]
            if type_dfs:
                years_available = sorted([int(df.split('_')[-1]) for df in type_dfs])
                print(f"    {file_type}: {min(years_available)}-{max(years_available)} ({len(years_available)} files)")

print(f"\n🎯 Ready for data cleaning and processing!")

📋 Station Configuration:
   Station 51028: 2005-2008 (4 years)
   Station 41008: 2006-2025 (17 years)

📁 File types to process: ['met', 'den', 'dir1', 'dir2', 'r1', 'r2']
📂 File mapping: {'met': 'stdmet', 'den': 'swden', 'dir1': 'swdir', 'dir2': 'swdir2', 'r1': 'swr1', 'r2': 'swr2'}
📂 Base data directory: ..\data\ndbc

🎯 Expected total files: 126

🚀 Starting dataframe creation...

📍 Processing Station 51028...
  📅 Year 2005: met✓ den✓ dir1✓ dir2✓ 

<>:63: SyntaxWarning: invalid escape sequence '\s'
<>:66: SyntaxWarning: invalid escape sequence '\s'
<>:63: SyntaxWarning: invalid escape sequence '\s'
<>:66: SyntaxWarning: invalid escape sequence '\s'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18044\3117273086.py:63: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(file_path, sep='\s+', skiprows=[1])
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18044\3117273086.py:66: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(file_path, sep='\s+')


r1✓ r2✓ 
  📅 Year 2006: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2007: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2008: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 

📍 Processing Station 41008...
  📅 Year 2006: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2007: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2008: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2009: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2013: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2014: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2015: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2016: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2017: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2018: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2019: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2020: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2021: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2022: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2023: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2024: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 
  📅 Year 2025: met✓ den✓ dir1✓ dir2✓ r1✓ r2✓ 

✅ Dataframe Creation Complete!
   📊 Tot

In [3]:
# 🔧 Standardize Column Names
# Handle variations: YYYY→YY, WD→WDIR
# Ensure met dataframes have: YY MM DD hh WDIR WSPD WVHT DPD APD MWD
# Ensure other dataframes have: YY MM DD hh

print("🔧 Standardizing column names...")

# Get list of all created dataframes
available_dfs = [name for name in globals().keys() if any(f"{ft}_" in name for ft in file_types) and any(station in name for station in stations_config.keys())]

standardized_count = 0
column_changes = {}

for df_name in available_dfs:
    try:
        df = globals()[df_name]
        changes_made = []
        
        # Create a mapping for column renaming
        column_mapping = {}
        
        # Handle year column: YYYY → YY (if YYYY exists and YY doesn't)
        if 'YYYY' in df.columns and 'YY' not in df.columns:
            column_mapping['YYYY'] = 'YY'
            changes_made.append('YYYY→YY')
        
        # Handle wind direction: WD → WDIR (if WD exists and WDIR doesn't)
        if 'WD' in df.columns and 'WDIR' not in df.columns:
            column_mapping['WD'] = 'WDIR'
            changes_made.append('WD→WDIR')
        
        # Apply column renaming if any changes needed
        if column_mapping:
            df = df.rename(columns=column_mapping)
            globals()[df_name] = df
            standardized_count += 1
            column_changes[df_name] = changes_made
        
    except Exception as e:
        print(f"  ❌ Error processing {df_name}: {str(e)}")

print(f"✅ Column standardization completed")
print(f"📊 Dataframes processed: {len(available_dfs)}")
print(f"🔧 Dataframes with changes: {standardized_count}")

if column_changes:
    print(f"\n📋 Column changes made:")
    for df_name, changes in column_changes.items():
        print(f"  {df_name}: {', '.join(changes)}")

    
print(f"\n🎯 Ready for timekey creation!")

🔧 Standardizing column names...
✅ Column standardization completed
📊 Dataframes processed: 126
🔧 Dataframes with changes: 18

📋 Column changes made:
  met_51028_2005: YYYY→YY, WD→WDIR
  den_51028_2005: YYYY→YY
  dir1_51028_2005: YYYY→YY
  dir2_51028_2005: YYYY→YY
  r1_51028_2005: YYYY→YY
  r2_51028_2005: YYYY→YY
  met_51028_2006: YYYY→YY, WD→WDIR
  den_51028_2006: YYYY→YY
  dir1_51028_2006: YYYY→YY
  dir2_51028_2006: YYYY→YY
  r1_51028_2006: YYYY→YY
  r2_51028_2006: YYYY→YY
  met_41008_2006: YYYY→YY, WD→WDIR
  den_41008_2006: YYYY→YY
  dir1_41008_2006: YYYY→YY
  dir2_41008_2006: YYYY→YY
  r1_41008_2006: YYYY→YY
  r2_41008_2006: YYYY→YY

🎯 Ready for timekey creation!


In [4]:
met_51028_2005.head()

,YY,MM,DD,hh,mm,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,BAR,ATMP,WTMP,DEWP,VIS,TIDE
0,2005,1,1,1,0,103,9.2,10.8,2.25,8.33,5.65,84,1003.8,27.1,27.1,999.0,99.0,99.0
1,2005,1,1,2,0,105,8.8,10.6,2.18,9.09,5.74,81,1003.7,27.1,27.1,999.0,99.0,99.0
2,2005,1,1,3,0,108,7.7,9.4,2.50,8.33,6.18,86,1004.1,27.2,27.1,999.0,99.0,99.0
3,2005,1,1,4,0,117,8.5,10.4,2.42,8.33,6.23,79,1004.6,27.2,27.1,999.0,99.0,99.0
4,2005,1,1,5,0,102,8.1,9.5,2.32,8.33,6.13,80,1005.1,27.2,27.0,999.0,99.0,99.0


In [5]:
met_51028_2007.head()

,YY,MM,DD,hh,mm,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,PRES,ATMP,WTMP,DEWP,VIS,TIDE
0,2007,1,1,0,0,111,6.9,8.2,99.0,99.0,99.0,999,1005.8,27.4,27.7,999.0,99.0,99.0
1,2007,1,1,1,0,115,8.4,9.9,99.0,99.0,99.0,999,1005.0,27.6,27.7,999.0,99.0,99.0
2,2007,1,1,2,0,97,9.2,11.3,99.0,99.0,99.0,999,1005.1,25.5,27.6,999.0,99.0,99.0
3,2007,1,1,3,0,105,6.4,7.7,99.0,99.0,99.0,999,1005.6,26.8,27.6,999.0,99.0,99.0
4,2007,1,1,4,0,116,9.4,11.0,99.0,99.0,99.0,999,1006.0,27.7,27.6,999.0,99.0,99.0


In [6]:
# 🔑 Add timekey to all dataframes
# Timekey = YY + MM + DD + hh (zero-padded)

print("🔑 Adding timekey to all dataframes...")

# Get list of all created dataframes
available_dfs = [name for name in globals().keys() if any(f"{ft}_" in name for ft in file_types) and any(station in name for station in stations_config.keys())]

timekey_added = 0
failed_dfs = []

for df_name in available_dfs:
    try:
        df = globals()[df_name]
        
        # Check if required time columns exist
        required_cols = ['YY', 'MM', 'DD', 'hh']
        if all(col in df.columns for col in required_cols):
            # Create timekey by joining YY, MM, DD, hh with zero padding
            df['timekey'] = (df['YY'].astype(str) + 
                           df['MM'].astype(str).str.zfill(2) + 
                           df['DD'].astype(str).str.zfill(2) + 
                           df['hh'].astype(str).str.zfill(2))
            
            # Make timekey the primary (first) column
            cols = ['timekey'] + [col for col in df.columns if col != 'timekey']
            df = df[cols]
            
            # Drop individual time columns
            time_cols_to_drop = ['YY', 'MM', 'DD', 'hh', 'mm']
            cols_to_drop = [col for col in time_cols_to_drop if col in df.columns]
            if cols_to_drop:
                df = df.drop(columns=cols_to_drop)
            
            globals()[df_name] = df
            timekey_added += 1
        else:
            print(f"  ❌ {df_name}: Missing time columns")
            failed_dfs.append(df_name)
            
    except Exception as e:
        print(f"  ❌ {df_name}: Error - {str(e)}")
        failed_dfs.append(df_name)

print(f"✅ Timekey added to {timekey_added} dataframes")
if failed_dfs:
    print(f"❌ Failed to process {len(failed_dfs)} dataframes: {failed_dfs}")


🔑 Adding timekey to all dataframes...
✅ Timekey added to 126 dataframes


In [7]:
met_51028_2005.head()

,timekey,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,BAR,ATMP,WTMP,DEWP,VIS,TIDE
0,2005010101,103,9.2,10.8,2.25,8.33,5.65,84,1003.8,27.1,27.1,999.0,99.0,99.0
1,2005010102,105,8.8,10.6,2.18,9.09,5.74,81,1003.7,27.1,27.1,999.0,99.0,99.0
2,2005010103,108,7.7,9.4,2.50,8.33,6.18,86,1004.1,27.2,27.1,999.0,99.0,99.0
3,2005010104,117,8.5,10.4,2.42,8.33,6.23,79,1004.6,27.2,27.1,999.0,99.0,99.0
4,2005010105,102,8.1,9.5,2.32,8.33,6.13,80,1005.1,27.2,27.0,999.0,99.0,99.0


In [8]:
# 📋 Filter met columns to keep only required meteorological variables
# Keep: WDIR, WSPD, WVHT, DPD, APD, MWD

print("📋 Filtering met dataframes to keep essential columns...")

# Define columns to keep (plus timekey and time components)
met_columns_to_keep = ['timekey', 'WDIR', 'WSPD', 'WVHT', 'DPD', 'APD', 'MWD']

# Filter all met dataframes
met_dataframes = [name for name in available_dfs if name.startswith('met_')]

for df_name in met_dataframes:
    df = globals()[df_name]
    # Keep only the specified columns that exist in the dataframe
    columns_to_keep = [col for col in met_columns_to_keep if col in df.columns]
    df_filtered = df[columns_to_keep].copy()
    globals()[df_name] = df_filtered

print(f"✅ Filtered {len(met_dataframes)} met dataframes")

📋 Filtering met dataframes to keep essential columns...
✅ Filtered 21 met dataframes


In [9]:
met_51028_2005.head()

,timekey,WDIR,WSPD,WVHT,DPD,APD,MWD
0,2005010101,103,9.2,2.25,8.33,5.65,84
1,2005010102,105,8.8,2.18,9.09,5.74,81
2,2005010103,108,7.7,2.50,8.33,6.18,86
3,2005010104,117,8.5,2.42,8.33,6.23,79
4,2005010105,102,8.1,2.32,8.33,6.13,80


In [10]:
# 🔄 Filter common timekeys across all file types for each station/year
# Ensure all dataframes have the same timekeys for proper alignment

print("🔄 Filtering for common timekeys across file types...")

# Process each station and year combination
for station, years in stations_config.items():
    for year in years:
        print(f"  📍 Processing Station {station}, Year {year}...")
        
        # Get all dataframes for this station/year
        station_year_dfs = []
        df_names = []
        
        for file_type in file_types:
            df_name = f"{file_type}_{station}_{year}"
            if df_name in globals():
                station_year_dfs.append(globals()[df_name])
                df_names.append(df_name)
        
        if len(station_year_dfs) > 1:
            # Find common timekeys across all dataframes
            common_timekeys = set(station_year_dfs[0]['timekey'])
            for df in station_year_dfs[1:]:
                common_timekeys = common_timekeys.intersection(set(df['timekey']))
            
            print(f"    🔑 Common timekeys: {len(common_timekeys)}")
            
            # Filter each dataframe to keep only common timekeys
            for i, df_name in enumerate(df_names):
                df_filtered = station_year_dfs[i][station_year_dfs[i]['timekey'].isin(common_timekeys)].copy()
                df_filtered = df_filtered.sort_values('timekey').reset_index(drop=True)
                globals()[df_name] = df_filtered
        
print(f"✅ Common timekey filtering completed")

🔄 Filtering for common timekeys across file types...
  📍 Processing Station 51028, Year 2005...
    🔑 Common timekeys: 6790
  📍 Processing Station 51028, Year 2006...
    🔑 Common timekeys: 7079
  📍 Processing Station 51028, Year 2007...
    🔑 Common timekeys: 2962
  📍 Processing Station 51028, Year 2008...
    🔑 Common timekeys: 2512
  📍 Processing Station 41008, Year 2006...
    🔑 Common timekeys: 739
  📍 Processing Station 41008, Year 2007...
    🔑 Common timekeys: 8652
  📍 Processing Station 41008, Year 2008...
    🔑 Common timekeys: 8708
  📍 Processing Station 41008, Year 2009...
    🔑 Common timekeys: 3287
  📍 Processing Station 41008, Year 2013...
    🔑 Common timekeys: 8582
  📍 Processing Station 41008, Year 2014...
    🔑 Common timekeys: 8456
  📍 Processing Station 41008, Year 2015...
    🔑 Common timekeys: 8672
  📍 Processing Station 41008, Year 2016...
    🔑 Common timekeys: 7269
  📍 Processing Station 41008, Year 2017...
    🔑 Common timekeys: 8653
  📍 Processing Station 41

In [11]:
# 🗑️ Remove duplicates from all dataframes using timekey

print("🗑️ Removing duplicates using timekey...")

duplicates_removed = 0

for df_name in available_dfs:
    if df_name in globals():
        df = globals()[df_name]
        original_shape = df.shape[0]
        
        # Remove duplicates based on timekey, keep first occurrence
        df_clean = df.drop_duplicates(subset=['timekey'], keep='first').reset_index(drop=True)
        
        duplicates_found = original_shape - df_clean.shape[0]
        duplicates_removed += duplicates_found
        
        globals()[df_name] = df_clean
        
        if duplicates_found > 0:
            print(f"  📋 {df_name}: Removed {duplicates_found} duplicates")

print(f"✅ Total duplicates removed: {duplicates_removed}")


🗑️ Removing duplicates using timekey...
  📋 met_51028_2008: Removed 1 duplicates
  📋 met_41008_2008: Removed 1 duplicates
  📋 met_41008_2023: Removed 21915 duplicates
  📋 den_41008_2023: Removed 4365 duplicates
  📋 dir1_41008_2023: Removed 4365 duplicates
  📋 dir2_41008_2023: Removed 4365 duplicates
  📋 r1_41008_2023: Removed 4365 duplicates
  📋 r2_41008_2023: Removed 4365 duplicates
  📋 met_41008_2024: Removed 42668 duplicates
  📋 den_41008_2024: Removed 8477 duplicates
  📋 dir1_41008_2024: Removed 8477 duplicates
  📋 dir2_41008_2024: Removed 8477 duplicates
  📋 r1_41008_2024: Removed 8477 duplicates
  📋 r2_41008_2024: Removed 8477 duplicates
  📋 met_41008_2025: Removed 43476 duplicates
  📋 den_41008_2025: Removed 8635 duplicates
  📋 dir1_41008_2025: Removed 8635 duplicates
  📋 dir2_41008_2025: Removed 8635 duplicates
  📋 r1_41008_2025: Removed 8635 duplicates
  📋 r2_41008_2025: Removed 8635 duplicates
✅ Total duplicates removed: 215446


In [12]:
r2_51028_2005.head()

,timekey,.0200,.0325,.0375,.0425,.0475,.0525,.0575,.0625,.0675,...,.3300,.3400,.3500,.3650,.3850,.4050,.4250,.4450,.4650,.4850
0,2005010101,20,22,53,25,44,18,39,19,7,...,50,15,47,52,23,50,8,22,26,12
1,2005010102,48,41,33,48,13,45,42,10,31,...,7,18,56,36,24,42,39,54,25,11
2,2005010103,44,35,15,14,20,31,19,14,20,...,33,21,70,42,17,22,22,45,29,30
3,2005010104,6,64,49,55,40,43,40,33,17,...,15,17,11,21,14,5,38,38,48,10
4,2005010105,16,32,12,9,36,25,44,37,34,...,27,53,10,29,23,26,27,18,53,37


In [13]:
# ⚖️ Scale r1 and r2 dataframes by 0.01

print("⚖️ Scaling r1 and r2 dataframes by 0.01...")

# Get all r1 and r2 dataframes
r1_r2_dataframes = [name for name in available_dfs if name.startswith(('r1_', 'r2_'))]

for df_name in r1_r2_dataframes:
    if df_name in globals():
        df = globals()[df_name]
        
        # Scale all numeric columns except time-related ones
        columns_to_scale = [col for col in df.columns if col not in ['timekey', 'YY', 'MM', 'DD', 'hh', 'mm']]
        
        for col in columns_to_scale:
            if df[col].dtype in ['int64', 'float64']:
                df[col] = df[col] * 0.01
        
        globals()[df_name] = df
        print(f"  📊 Scaled {df_name}: {len(columns_to_scale)} columns")

print(f"✅ Scaling completed for {len(r1_r2_dataframes)} dataframes")

⚖️ Scaling r1 and r2 dataframes by 0.01...
  📊 Scaled r1_51028_2005: 47 columns
  📊 Scaled r2_51028_2005: 47 columns


  📊 Scaled r1_51028_2006: 47 columns
  📊 Scaled r2_51028_2006: 47 columns
  📊 Scaled r1_51028_2007: 47 columns
  📊 Scaled r2_51028_2007: 47 columns
  📊 Scaled r1_51028_2008: 47 columns
  📊 Scaled r2_51028_2008: 47 columns
  📊 Scaled r1_41008_2006: 47 columns
  📊 Scaled r2_41008_2006: 47 columns
  📊 Scaled r1_41008_2007: 47 columns
  📊 Scaled r2_41008_2007: 47 columns
  📊 Scaled r1_41008_2008: 47 columns
  📊 Scaled r2_41008_2008: 47 columns
  📊 Scaled r1_41008_2009: 47 columns
  📊 Scaled r2_41008_2009: 47 columns
  📊 Scaled r1_41008_2013: 47 columns
  📊 Scaled r2_41008_2013: 47 columns
  📊 Scaled r1_41008_2014: 47 columns
  📊 Scaled r2_41008_2014: 47 columns
  📊 Scaled r1_41008_2015: 47 columns
  📊 Scaled r2_41008_2015: 47 columns
  📊 Scaled r1_41008_2016: 47 columns
  📊 Scaled r2_41008_2016: 47 columns
  📊 Scaled r1_41008_2017: 47 columns
  📊 Scaled r2_41008_2017: 47 columns
  📊 Scaled r1_41008_2018: 47 columns
  📊 Scaled r2_41008_2018: 47 columns
  📊 Scaled r1_41008_2019: 47 columns
 

In [14]:
r2_51028_2005.head()

,timekey,.0200,.0325,.0375,.0425,.0475,.0525,.0575,.0625,.0675,...,.3300,.3400,.3500,.3650,.3850,.4050,.4250,.4450,.4650,.4850
0,2005010101,0.20,0.22,0.53,0.25,0.44,0.18,0.39,0.19,0.07,...,0.50,0.15,0.47,0.52,0.23,0.50,0.08,0.22,0.26,0.12
1,2005010102,0.48,0.41,0.33,0.48,0.13,0.45,0.42,0.10,0.31,...,0.07,0.18,0.56,0.36,0.24,0.42,0.39,0.54,0.25,0.11
2,2005010103,0.44,0.35,0.15,0.14,0.20,0.31,0.19,0.14,0.20,...,0.33,0.21,0.70,0.42,0.17,0.22,0.22,0.45,0.29,0.30
3,2005010104,0.06,0.64,0.49,0.55,0.40,0.43,0.40,0.33,0.17,...,0.15,0.17,0.11,0.21,0.14,0.05,0.38,0.38,0.48,0.10
4,2005010105,0.16,0.32,0.12,0.09,0.36,0.25,0.44,0.37,0.34,...,0.27,0.53,0.10,0.29,0.23,0.26,0.27,0.18,0.53,0.37


In [15]:
# 🚫 Convert nodata values to NaN (with selective 99 handling)
# Nodata values: 999, 9999 (always), 99 (selectively)
# 99 is ACCEPTED for: met(WDIR,MWD), dir1(all), dir2(all), r1(all), r2(all)

print("🚫 Converting nodata values to NaN...")

# Define nodata values
always_nodata = [999, 9999]  # Always convert these
selective_nodata = 99  # Only convert in specific cases

print(f"🎯 Always nodata: {always_nodata}")
print(f"🎯 Selective nodata (99): Only for specific dataframes/columns")

total_conversions = 0

# Update available dataframes list
available_dfs = [name for name in globals().keys() if any(f"{ft}_" in name for ft in file_types) and any(station in name for station in stations_config.keys())]

for df_name in available_dfs:
    if df_name in globals():
        df = globals()[df_name]
        
        # Get dataframe type and process accordingly
        df_type = df_name.split('_')[0]  # met, den, dir1, dir2, r1, r2
        
        # Convert always_nodata values for all numeric columns except timekey
        numeric_columns = [col for col in df.columns if col != 'timekey' and df[col].dtype in ['int64', 'float64']]
        
        conversions_this_df = 0
        
        # 1. Always convert 999 and 9999 to NaN for all dataframes
        for col in numeric_columns:
            for nodata_val in always_nodata:
                mask = df[col] == nodata_val
                conversions_count = mask.sum()
                if conversions_count > 0:
                    df.loc[mask, col] = np.nan
                    conversions_this_df += conversions_count
        
        # 2. Selectively convert 99 to NaN based on dataframe type and column
        for col in numeric_columns:
            should_convert_99 = False
            
            if df_type == 'met':
                # For met dataframes: convert 99 to NaN EXCEPT for WDIR and MWD
                if col not in ['WDIR', 'MWD']:
                    should_convert_99 = True
            elif df_type in ['dir1', 'dir2', 'r1', 'r2']:
                # For dir1/dir2/r1/r2 dataframes: 99 is accepted (don't convert)
                should_convert_99 = False
            else:
                # For den dataframes: convert 99 to NaN
                should_convert_99 = True
            
            if should_convert_99:
                mask = df[col] == selective_nodata
                conversions_count = mask.sum()
                if conversions_count > 0:
                    df.loc[mask, col] = np.nan
                    conversions_this_df += conversions_count
        
        total_conversions += conversions_this_df
        globals()[df_name] = df
        
        if conversions_this_df > 0:
            print(f"  🔄 {df_name}: {conversions_this_df} values converted")

print(f"✅ Total nodata values converted to NaN: {total_conversions}")

print(f"\n📋 99 Value Handling:")
print(f"  ✅ KEPT as valid data in: met(WDIR,MWD), dir1(all), dir2(all), r1(all), r2(all)")
print(f"  🚫 CONVERTED to NaN in: met(other cols), den(all)")



🚫 Converting nodata values to NaN...
🎯 Always nodata: [999, 9999]
🎯 Selective nodata (99): Only for specific dataframes/columns
  🔄 met_51028_2005: 4 values converted
  🔄 met_51028_2006: 8 values converted
  🔄 met_51028_2008: 20 values converted
  🔄 r1_41008_2007: 2 values converted
  🔄 met_41008_2008: 190 values converted
  🔄 r1_41008_2008: 3 values converted
  🔄 r2_41008_2008: 1 values converted
  🔄 met_41008_2009: 121 values converted
  🔄 met_41008_2013: 1104 values converted
  🔄 met_41008_2014: 142 values converted
  🔄 met_41008_2015: 82 values converted
  🔄 met_41008_2016: 2266 values converted
  🔄 met_41008_2017: 172 values converted
  🔄 met_41008_2018: 68 values converted
  🔄 met_41008_2019: 92 values converted
  🔄 met_41008_2020: 110 values converted
  🔄 met_41008_2021: 142 values converted
  🔄 met_41008_2022: 76 values converted
  🔄 met_41008_2023: 12182 values converted
  🔄 met_41008_2024: 26836 values converted
  🔄 met_41008_2025: 28392 values converted
✅ Total nodata values

In [16]:
# 📊 NaN Analysis Summary for Met DataFrames

print("📊 NaN Analysis Summary for Met DataFrames")
print("=" * 50)

# Get all met dataframes
met_dataframes = [name for name in globals().keys() if name.startswith('met_') and isinstance(globals()[name], pd.DataFrame)]

for df_name in met_dataframes:
    df = globals()[df_name]
    total_rows = len(df)
    
    print(f"\n📁 File: {df_name}")
    print(f"📊 Total row count: {total_rows:,}")
    
    # Calculate NaN percentages for each column
    print(f"📋 Column NaN percentages:")
    
    for col in df.columns:
        if col != 'timekey':  # Skip timekey column
            nan_count = df[col].isna().sum()
            nan_percentage = (nan_count / total_rows) * 100
            print(f"   {col}: {nan_percentage:.2f}%")
    
    print("-" * 40)

print(f"\n✅ Analysis complete for {len(met_dataframes)} met dataframes")

📊 NaN Analysis Summary for Met DataFrames

📁 File: met_51028_2005
📊 Total row count: 6,790
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.03%
   APD: 0.03%
   MWD: 0.00%
----------------------------------------

📁 File: met_51028_2006
📊 Total row count: 7,079
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.03%
   APD: 0.00%
   MWD: 0.08%
----------------------------------------

📁 File: met_51028_2007
📊 Total row count: 2,962
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%
   MWD: 0.00%
----------------------------------------

📁 File: met_51028_2008
📊 Total row count: 2,512
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.20%
   DPD: 0.20%
   APD: 0.20%
   MWD: 0.20%
----------------------------------------

📁 File: met_41008_2006
📊 Total row count: 739
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%


In [17]:
# 🧹 Clean Met Data for Stations 51028 and 41008
# Drop rows with NaNs in critical columns: WVHT, DPD, APD, WSPD, WDIR, MWD

print("🧹 Cleaning met data for stations 51028 and 41008...")
print("🎯 Dropping rows with NaNs in: WVHT, DPD, APD, WSPD, WDIR, MWD")

# Define target stations and critical columns
target_stations = ['51028', '41008']
critical_columns = ['WVHT', 'DPD', 'APD', 'WSPD', 'WDIR', 'MWD']  # Include WDIR as well

# Get met dataframes for target stations only
met_dataframes = [name for name in globals().keys() 
                 if name.startswith('met_') 
                 and any(station in name for station in target_stations)
                 and isinstance(globals()[name], pd.DataFrame)]

total_rows_before = 0
total_rows_after = 0
cleaned_count = 0

print(f"\n📋 Processing {len(met_dataframes)} met dataframes...")

for df_name in met_dataframes:
    df = globals()[df_name]
    rows_before = len(df)
    total_rows_before += rows_before
    
    # Check which critical columns exist in this dataframe
    cols_to_check = [col for col in critical_columns if col in df.columns]
    
    if cols_to_check:
        # Drop rows where ANY of the critical columns have NaN values
        df_clean = df.dropna(subset=cols_to_check).reset_index(drop=True)
        rows_after = len(df_clean)
        rows_dropped = rows_before - rows_after
        
        # Update the dataframe
        globals()[df_name] = df_clean
        total_rows_after += rows_after
        cleaned_count += 1
        
        print(f"  📁 {df_name}: {rows_before:,} → {rows_after:,} rows ({rows_dropped:,} dropped)")
    else:
        print(f"  ⚠️  {df_name}: No critical columns found, keeping {rows_before:,} rows")
        total_rows_after += rows_before

total_dropped = total_rows_before - total_rows_after
drop_percentage = (total_dropped / total_rows_before) * 100 if total_rows_before > 0 else 0

print(f"\n✅ Met data cleaning completed!")
print(f"   📊 Total rows before: {total_rows_before:,}")
print(f"   📊 Total rows after: {total_rows_after:,}")
print(f"   🗑️  Total rows dropped: {total_dropped:,} ({drop_percentage:.2f}%)")
print(f"   📋 Dataframes cleaned: {cleaned_count}")



🧹 Cleaning met data for stations 51028 and 41008...
🎯 Dropping rows with NaNs in: WVHT, DPD, APD, WSPD, WDIR, MWD

📋 Processing 21 met dataframes...
  📁 met_51028_2005: 6,790 → 6,788 rows (2 dropped)
  📁 met_51028_2006: 7,079 → 7,073 rows (6 dropped)
  📁 met_51028_2007: 2,962 → 2,962 rows (0 dropped)
  📁 met_51028_2008: 2,512 → 2,507 rows (5 dropped)
  📁 met_41008_2006: 739 → 739 rows (0 dropped)
  📁 met_41008_2007: 8,652 → 8,652 rows (0 dropped)
  📁 met_41008_2008: 8,708 → 8,656 rows (52 dropped)
  📁 met_41008_2009: 3,287 → 3,250 rows (37 dropped)
  📁 met_41008_2013: 8,582 → 8,041 rows (541 dropped)
  📁 met_41008_2014: 8,456 → 8,406 rows (50 dropped)
  📁 met_41008_2015: 8,672 → 8,644 rows (28 dropped)
  📁 met_41008_2016: 7,269 → 6,151 rows (1,118 dropped)
  📁 met_41008_2017: 8,653 → 8,583 rows (70 dropped)
  📁 met_41008_2018: 8,637 → 8,613 rows (24 dropped)
  📁 met_41008_2019: 8,616 → 8,586 rows (30 dropped)
  📁 met_41008_2020: 8,623 → 8,580 rows (43 dropped)
  📁 met_41008_2021: 8,551

In [18]:
# 📊 NaN Analysis Summary for Met DataFrames

print("📊 NaN Analysis Summary for Met DataFrames")
print("=" * 50)

# Get all met dataframes
met_dataframes = [name for name in globals().keys() if name.startswith('met_') and isinstance(globals()[name], pd.DataFrame)]

for df_name in met_dataframes:
    df = globals()[df_name]
    total_rows = len(df)
    
    print(f"\n📁 File: {df_name}")
    print(f"📊 Total row count: {total_rows:,}")
    
    # Calculate NaN percentages for each column
    print(f"📋 Column NaN percentages:")
    
    for col in df.columns:
        if col != 'timekey':  # Skip timekey column
            nan_count = df[col].isna().sum()
            nan_percentage = (nan_count / total_rows) * 100
            print(f"   {col}: {nan_percentage:.2f}%")
    
    print("-" * 40)

print(f"\n✅ Analysis complete for {len(met_dataframes)} met dataframes")

📊 NaN Analysis Summary for Met DataFrames

📁 File: met_51028_2005
📊 Total row count: 6,788
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%
   MWD: 0.00%
----------------------------------------

📁 File: met_51028_2006
📊 Total row count: 7,073
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%
   MWD: 0.00%
----------------------------------------

📁 File: met_51028_2007
📊 Total row count: 2,962
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%
   MWD: 0.00%
----------------------------------------

📁 File: met_51028_2008
📊 Total row count: 2,507
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%
   MWD: 0.00%
----------------------------------------

📁 File: met_41008_2006
📊 Total row count: 739
📋 Column NaN percentages:
   WDIR: 0.00%
   WSPD: 0.00%
   WVHT: 0.00%
   DPD: 0.00%
   APD: 0.00%


In [19]:
# 📊 Total Row Count for Met Data (Stations 51028 and 41008)

print("📊 Total Row Count for Met Data")
print("=" * 40)

# Target stations
target_stations = ['51028', '41008']

# Get met dataframes for target stations only
met_dataframes = [name for name in globals().keys() 
                 if name.startswith('met_') 
                 and any(station in name for station in target_stations)
                 and isinstance(globals()[name], pd.DataFrame)]

total_rows = 0
station_totals = {'51028': 0, '41008': 0}

print(f"📋 Met dataframes found: {len(met_dataframes)}")

for df_name in met_dataframes:
    df = globals()[df_name]
    rows = len(df)
    total_rows += rows
    
    # Determine station
    station = df_name.split('_')[1]
    if station in station_totals:
        station_totals[station] += rows
    
    print(f"  📁 {df_name}: {rows:,} rows")

print(f"\n📊 Station Totals:")
for station, count in station_totals.items():
    print(f"  Station {station}: {count:,} rows")

print(f"\n🎯 TOTAL ROWS (both stations): {total_rows:,}")

📊 Total Row Count for Met Data
📋 Met dataframes found: 21
  📁 met_51028_2005: 6,788 rows
  📁 met_51028_2006: 7,073 rows
  📁 met_51028_2007: 2,962 rows
  📁 met_51028_2008: 2,507 rows
  📁 met_41008_2006: 739 rows
  📁 met_41008_2007: 8,652 rows
  📁 met_41008_2008: 8,656 rows
  📁 met_41008_2009: 3,250 rows
  📁 met_41008_2013: 8,041 rows
  📁 met_41008_2014: 8,406 rows
  📁 met_41008_2015: 8,644 rows
  📁 met_41008_2016: 6,151 rows
  📁 met_41008_2017: 8,583 rows
  📁 met_41008_2018: 8,613 rows
  📁 met_41008_2019: 8,586 rows
  📁 met_41008_2020: 8,580 rows
  📁 met_41008_2021: 8,490 rows
  📁 met_41008_2022: 8,655 rows
  📁 met_41008_2023: 5,322 rows
  📁 met_41008_2024: 1,825 rows
  📁 met_41008_2025: 1,791 rows

📊 Station Totals:
  Station 51028: 19,330 rows
  Station 41008: 112,984 rows

🎯 TOTAL ROWS (both stations): 132,314


In [20]:
# 📊 NaN Analysis for Spectral Data (Stations 51028 and 41008)
# Dataframes: den, dir1, dir2, r1, r2

print("📊 NaN Analysis for Spectral Data (Stations 51028 and 41008)")
print("=" * 60)

# Target stations and dataframe types
target_stations = ['51028', '41008']
spectral_types = ['den', 'dir1', 'dir2', 'r1', 'r2']

# Get spectral dataframes for target stations only
spectral_dataframes = []
for df_type in spectral_types:
    for df_name in globals().keys():
        if (df_name.startswith(f'{df_type}_') and 
            any(station in df_name for station in target_stations) and
            isinstance(globals()[df_name], pd.DataFrame)):
            spectral_dataframes.append(df_name)

spectral_dataframes.sort()

for df_name in spectral_dataframes:
    df = globals()[df_name]
    total_rows = len(df)
    
    print(f"\n📁 File: {df_name}")
    print(f"📊 Total row count: {total_rows:,}")
    
    # Count rows with any NaN values (excluding timekey column)
    data_columns = [col for col in df.columns if col != 'timekey']
    rows_with_nan = df[data_columns].isna().any(axis=1).sum()
    nan_percentage = (rows_with_nan / total_rows) * 100
    
    print(f"📋 Rows with any NaN: {rows_with_nan:,} ({nan_percentage:.2f}%)")
    print("-" * 50)

print(f"\n✅ Analysis complete for {len(spectral_dataframes)} spectral dataframes")

📊 NaN Analysis for Spectral Data (Stations 51028 and 41008)

📁 File: den_41008_2006
📊 Total row count: 739
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2007
📊 Total row count: 8,652
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2008
📊 Total row count: 8,708
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2009
📊 Total row count: 3,287
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2013
📊 Total row count: 8,582
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2014
📊 Total row count: 8,456
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2015
📊 Total row count: 8,672
📋 Rows with any NaN: 0 (0.00%)
--------------------------------------------------

📁 File: den_41008_2016
📊

In [21]:
# 🧹 Handle NaN Values in Spectral Data (Stations 51028 and 41008)
# Drop rows with NaNs for all spectral data types: den, dir1, dir2, r1, r2

print("🧹 Handling NaN values in spectral data...")
print("🎯 Dropping rows with NaNs for all spectral data types")

# Target stations and spectral data types
target_stations = ['51028', '41008']
spectral_types = ['den', 'dir1', 'dir2', 'r1', 'r2']

total_rows_before = 0
total_rows_after = 0
total_rows_dropped = 0

print(f"\n📋 Processing spectral dataframes for stations: {target_stations}")

# Process each spectral data type
for df_type in spectral_types:
    print(f"\n🗑️ Processing {df_type} dataframes...")
    
    # Collect dataframe names for this type and target stations
    dataframes_to_process = [df_name for df_name in globals().keys()
                            if (df_name.startswith(f'{df_type}_') and 
                                any(station in df_name for station in target_stations) and
                                isinstance(globals()[df_name], pd.DataFrame))]
    
    for df_name in dataframes_to_process:
        df = globals()[df_name]
        rows_before = len(df)
        total_rows_before += rows_before
        
        # Drop rows with any NaN values (excluding timekey)
        data_cols = [col for col in df.columns if col != 'timekey']
        df_clean = df.dropna(subset=data_cols).reset_index(drop=True)
        
        rows_after = len(df_clean)
        rows_dropped = rows_before - rows_after
        total_rows_dropped += rows_dropped
        total_rows_after += rows_after
        
        # Update the dataframe
        globals()[df_name] = df_clean
        
        if rows_dropped > 0:
            print(f"  📁 {df_name}: {rows_before:,} → {rows_after:,} ({rows_dropped:,} dropped)")
        else:
            print(f"  📁 {df_name}: {rows_before:,} rows (no NaNs found)")

drop_percentage = (total_rows_dropped / total_rows_before) * 100 if total_rows_before > 0 else 0

print(f"\n✅ Spectral data NaN handling completed!")
print(f"   📊 Total rows before: {total_rows_before:,}")
print(f"   📊 Total rows after: {total_rows_after:,}")
print(f"   🗑️ Total rows dropped: {total_rows_dropped:,} ({drop_percentage:.2f}%)")
print(f"   📋 Dataframe types processed: {spectral_types}")


🧹 Handling NaN values in spectral data...
🎯 Dropping rows with NaNs for all spectral data types

📋 Processing spectral dataframes for stations: ['51028', '41008']

🗑️ Processing den dataframes...
  📁 den_51028_2005: 6,790 rows (no NaNs found)
  📁 den_51028_2006: 7,079 rows (no NaNs found)
  📁 den_51028_2007: 2,962 rows (no NaNs found)
  📁 den_51028_2008: 2,512 rows (no NaNs found)
  📁 den_41008_2006: 739 rows (no NaNs found)
  📁 den_41008_2007: 8,652 rows (no NaNs found)
  📁 den_41008_2008: 8,708 rows (no NaNs found)
  📁 den_41008_2009: 3,287 rows (no NaNs found)
  📁 den_41008_2013: 8,582 rows (no NaNs found)
  📁 den_41008_2014: 8,456 rows (no NaNs found)
  📁 den_41008_2015: 8,672 rows (no NaNs found)
  📁 den_41008_2016: 7,269 rows (no NaNs found)
  📁 den_41008_2017: 8,653 rows (no NaNs found)
  📁 den_41008_2018: 8,637 rows (no NaNs found)
  📁 den_41008_2019: 8,616 rows (no NaNs found)
  📁 den_41008_2020: 8,623 rows (no NaNs found)
  📁 den_41008_2021: 8,551 rows (no NaNs found)
  📁 den

In [22]:
# 📊 Total Row Count for Met Data (Stations 51028 and 41008)

print("📊 Total Row Count for Met Data")
print("=" * 40)

# Target stations
target_stations = ['51028', '41008']

# Get met dataframes for target stations only
met_dataframes = [name for name in globals().keys() 
                 if name.startswith('met_') 
                 and any(station in name for station in target_stations)
                 and isinstance(globals()[name], pd.DataFrame)]

total_rows = 0
station_totals = {'51028': 0, '41008': 0}

print(f"📋 Met dataframes found: {len(met_dataframes)}")

for df_name in met_dataframes:
    df = globals()[df_name]
    rows = len(df)
    total_rows += rows
    
    # Determine station
    station = df_name.split('_')[1]
    if station in station_totals:
        station_totals[station] += rows
    
    print(f"  📁 {df_name}: {rows:,} rows")

print(f"\n📊 Station Totals:")
for station, count in station_totals.items():
    print(f"  Station {station}: {count:,} rows")

print(f"\n🎯 TOTAL ROWS (both stations): {total_rows:,}")

📊 Total Row Count for Met Data
📋 Met dataframes found: 21
  📁 met_51028_2005: 6,788 rows
  📁 met_51028_2006: 7,073 rows
  📁 met_51028_2007: 2,962 rows
  📁 met_51028_2008: 2,507 rows
  📁 met_41008_2006: 739 rows
  📁 met_41008_2007: 8,652 rows
  📁 met_41008_2008: 8,656 rows
  📁 met_41008_2009: 3,250 rows
  📁 met_41008_2013: 8,041 rows
  📁 met_41008_2014: 8,406 rows
  📁 met_41008_2015: 8,644 rows
  📁 met_41008_2016: 6,151 rows
  📁 met_41008_2017: 8,583 rows
  📁 met_41008_2018: 8,613 rows
  📁 met_41008_2019: 8,586 rows
  📁 met_41008_2020: 8,580 rows
  📁 met_41008_2021: 8,490 rows
  📁 met_41008_2022: 8,655 rows
  📁 met_41008_2023: 5,322 rows
  📁 met_41008_2024: 1,825 rows
  📁 met_41008_2025: 1,791 rows

📊 Station Totals:
  Station 51028: 19,330 rows
  Station 41008: 112,984 rows

🎯 TOTAL ROWS (both stations): 132,314


In [23]:
# 🔒 Enforce Physical Bounds on Meteorological Data
# Drop rows that violate oceanographic and meteorological constraints

print("🔒 Enforcing physical bounds on meteorological data...")

# Define physical bounds constraints
physical_bounds = {
    'WSPD': {'min': 0, 'description': 'wind speed must be non-negative'},
    'WVHT': {'min': 0, 'description': 'wave height must be non-negative'},
    'DPD': {'min': 0, 'min_exclusive': True, 'description': 'dominant period must be positive'},
    'APD': {'min': 0, 'min_exclusive': True, 'description': 'average period must be positive'},
    'WDIR': {'min': 0, 'max': 360, 'description': 'wind direction valid compass range [0, 360]'},
    'MWD': {'min': 0, 'max': 360, 'description': 'mean wave direction valid compass range [0, 360]'}
}

# Target stations
target_stations = ['51028', '41008']

# Get all met dataframes for target stations
met_dataframes = [name for name in globals().keys() 
                 if name.startswith('met_') 
                 and any(station in name for station in target_stations)
                 and isinstance(globals()[name], pd.DataFrame)]

print(f"🎯 Found {len(met_dataframes)} met dataframes to validate")
print(f"📋 Physical bounds constraints:")
for param, bounds in physical_bounds.items():
    if 'max' in bounds:
        constraint = f"[{bounds['min']}, {bounds['max']}]"
    elif bounds.get('min_exclusive'):
        constraint = f"> {bounds['min']}"
    else:
        constraint = f">= {bounds['min']}"
    print(f"   {param}: {constraint} - {bounds['description']}")

total_rows_before = 0
total_rows_after = 0
total_violations = 0
violation_summary = {}

print(f"\n🔍 Validating physical bounds...")

for df_name in met_dataframes:
    df = globals()[df_name]
    rows_before = len(df)
    total_rows_before += rows_before
    
    # Initialize valid row mask (start with all True)
    valid_mask = pd.Series([True] * len(df), index=df.index)
    violations_this_df = {}
    
    # Check each physical constraint
    for param, bounds in physical_bounds.items():
        if param in df.columns:
            param_violations = 0
            
            # Apply minimum constraint
            if 'min' in bounds:
                if bounds.get('min_exclusive'):
                    # Exclusive constraint (greater than)
                    violation_mask = df[param] <= bounds['min']
                else:
                    # Inclusive constraint (greater than or equal)
                    violation_mask = df[param] < bounds['min']
                
                param_violations += violation_mask.sum()
                valid_mask &= ~violation_mask
            
            # Apply maximum constraint  
            if 'max' in bounds:
                violation_mask = df[param] > bounds['max']
                param_violations += violation_mask.sum()
                valid_mask &= ~violation_mask
            
            if param_violations > 0:
                violations_this_df[param] = param_violations
                if param not in violation_summary:
                    violation_summary[param] = 0
                violation_summary[param] += param_violations
    
    # Apply the valid mask to keep only physically valid rows
    df_clean = df[valid_mask].reset_index(drop=True)
    rows_after = len(df_clean)
    rows_dropped = rows_before - rows_after
    total_rows_after += rows_after
    total_violations += rows_dropped
    
    # Update the dataframe
    globals()[df_name] = df_clean
    
    # Report results for this dataframe
    if rows_dropped > 0:
        print(f"  📁 {df_name}: {rows_before:,} → {rows_after:,} ({rows_dropped:,} dropped)")
        for param, count in violations_this_df.items():
            print(f"    - {param}: {count:,} violations")
    else:
        print(f"  ✅ {df_name}: {rows_before:,} rows (no violations)")

drop_percentage = (total_violations / total_rows_before) * 100 if total_rows_before > 0 else 0

print(f"\n✅ Physical bounds validation completed!")
print(f"   📊 Total rows before: {total_rows_before:,}")
print(f"   📊 Total rows after: {total_rows_after:,}")
print(f"   🗑️ Total violations removed: {total_violations:,} ({drop_percentage:.2f}%)")
print(f"   📋 Dataframes processed: {len(met_dataframes)}")

# Summary of violations by parameter
if violation_summary:
    print(f"\n📋 Violations by parameter:")
    for param, count in violation_summary.items():
        print(f"   {param}: {count:,} violations")

# Verify data quality after bounds enforcement
print(f"\n🔍 Post-validation data ranges:")
sample_df_name = met_dataframes[0] if met_dataframes else None
if sample_df_name:
    sample_df = globals()[sample_df_name]
    for param in physical_bounds.keys():
        if param in sample_df.columns:
            min_val = sample_df[param].min()
            max_val = sample_df[param].max()
            print(f"   {param}: [{min_val:.2f}, {max_val:.2f}]")

print(f"\n🎉 All meteorological data now satisfies physical constraints!")


🔒 Enforcing physical bounds on meteorological data...
🎯 Found 21 met dataframes to validate
📋 Physical bounds constraints:
   WSPD: >= 0 - wind speed must be non-negative
   WVHT: >= 0 - wave height must be non-negative
   DPD: > 0 - dominant period must be positive
   APD: > 0 - average period must be positive
   WDIR: [0, 360] - wind direction valid compass range [0, 360]
   MWD: [0, 360] - mean wave direction valid compass range [0, 360]

🔍 Validating physical bounds...
  📁 met_51028_2005: 6,788 → 6,787 (1 dropped)
    - DPD: 1 violations
    - APD: 1 violations
  ✅ met_51028_2006: 7,073 rows (no violations)
  ✅ met_51028_2007: 2,962 rows (no violations)
  ✅ met_51028_2008: 2,507 rows (no violations)
  ✅ met_41008_2006: 739 rows (no violations)
  ✅ met_41008_2007: 8,652 rows (no violations)
  📁 met_41008_2008: 8,656 → 8,654 (2 dropped)
    - MWD: 2 violations
  ✅ met_41008_2009: 3,250 rows (no violations)
  ✅ met_41008_2013: 8,041 rows (no violations)
  ✅ met_41008_2014: 8,406 rows 

In [24]:
# 🔄 Filter common timekeys across all file types for each station/year
# Ensure all dataframes have the same timekeys for proper alignment

print("🔄 Filtering for common timekeys across file types...")

# Process each station and year combination
for station, years in stations_config.items():
    for year in years:
        print(f"  📍 Processing Station {station}, Year {year}...")
        
        # Get all dataframes for this station/year
        station_year_dfs = []
        df_names = []
        
        for file_type in file_types:
            df_name = f"{file_type}_{station}_{year}"
            if df_name in globals():
                station_year_dfs.append(globals()[df_name])
                df_names.append(df_name)
        
        if len(station_year_dfs) > 1:
            # Find common timekeys across all dataframes
            common_timekeys = set(station_year_dfs[0]['timekey'])
            for df in station_year_dfs[1:]:
                common_timekeys = common_timekeys.intersection(set(df['timekey']))
            
            print(f"    🔑 Common timekeys: {len(common_timekeys)}")
            
            # Filter each dataframe to keep only common timekeys
            for i, df_name in enumerate(df_names):
                df_filtered = station_year_dfs[i][station_year_dfs[i]['timekey'].isin(common_timekeys)].copy()
                df_filtered = df_filtered.sort_values('timekey').reset_index(drop=True)
                globals()[df_name] = df_filtered
        
print(f"✅ Common timekey filtering completed")


🔄 Filtering for common timekeys across file types...
  📍 Processing Station 51028, Year 2005...
    🔑 Common timekeys: 6787
  📍 Processing Station 51028, Year 2006...
    🔑 Common timekeys: 7073
  📍 Processing Station 51028, Year 2007...
    🔑 Common timekeys: 2962
  📍 Processing Station 51028, Year 2008...
    🔑 Common timekeys: 2507
  📍 Processing Station 41008, Year 2006...
    🔑 Common timekeys: 739
  📍 Processing Station 41008, Year 2007...
    🔑 Common timekeys: 8650
  📍 Processing Station 41008, Year 2008...
    🔑 Common timekeys: 8650
  📍 Processing Station 41008, Year 2009...
    🔑 Common timekeys: 3250
  📍 Processing Station 41008, Year 2013...
    🔑 Common timekeys: 8041
  📍 Processing Station 41008, Year 2014...
    🔑 Common timekeys: 8406
  📍 Processing Station 41008, Year 2015...
    🔑 Common timekeys: 8644
  📍 Processing Station 41008, Year 2016...
    🔑 Common timekeys: 6151
  📍 Processing Station 41008, Year 2017...
    🔑 Common timekeys: 8583
  📍 Processing Station 41

In [30]:
# 🔒 Verify R1 and R2 Correlation Coefficients Bounds
# Ensure r1 and r2 values are between 0 and 1 for all spectral dataframes

print("🔒 Verifying r1 and r2 correlation coefficient bounds...")

# Target stations and r1/r2 dataframe types
target_stations = ['51028', '41008']
r_types = ['r1', 'r2']

# Get all r1 and r2 dataframes for target stations
r_dataframes = []
for r_type in r_types:
    for df_name in globals().keys():
        if (df_name.startswith(f'{r_type}_') and 
            any(station in df_name for station in target_stations) and
            isinstance(globals()[df_name], pd.DataFrame)):
            r_dataframes.append(df_name)

r_dataframes.sort()

print(f"🎯 Found {len(r_dataframes)} r1/r2 dataframes to verify")
print(f"📋 Expected bounds: r1, r2 ∈ [0, 1] (correlation coefficients)")

total_rows_before = 0
total_rows_after = 0 
total_violations = 0
violation_summary = {}

print(f"\n🔍 Validating r1/r2 bounds...")

for df_name in r_dataframes:
    df = globals()[df_name]
    rows_before = len(df)
    total_rows_before += rows_before
    
    # Get numeric columns (exclude timekey)
    numeric_columns = [col for col in df.columns if col != 'timekey' and df[col].dtype in ['int64', 'float64']]
    
    # Initialize valid row mask (start with all True)
    valid_mask = pd.Series([True] * len(df), index=df.index)
    violations_this_df = 0
    
    # Check bounds for each numeric column
    for col in numeric_columns:
        # Check lower bound: values < 0
        lower_violations = (df[col] < 0).sum()
        if lower_violations > 0:
            valid_mask &= ~(df[col] < 0)
            violations_this_df += lower_violations
        
        # Check upper bound: values > 1  
        upper_violations = (df[col] > 1).sum()
        if upper_violations > 0:
            valid_mask &= ~(df[col] > 1)
            violations_this_df += upper_violations
        
        # Track total violations per column type
        total_col_violations = lower_violations + upper_violations
        if total_col_violations > 0:
            r_type = df_name.split('_')[0]  # r1 or r2
            if r_type not in violation_summary:
                violation_summary[r_type] = {'lower': 0, 'upper': 0, 'total': 0}
            violation_summary[r_type]['lower'] += lower_violations
            violation_summary[r_type]['upper'] += upper_violations
            violation_summary[r_type]['total'] += total_col_violations
    
    # Apply the valid mask to keep only properly bounded rows
    df_clean = df[valid_mask].reset_index(drop=True)
    rows_after = len(df_clean)
    rows_dropped = rows_before - rows_after
    total_rows_after += rows_after
    total_violations += rows_dropped
    
    # Update the dataframe
    globals()[df_name] = df_clean
    
    # Report results for this dataframe
    if rows_dropped > 0:
        print(f"  📁 {df_name}: {rows_before:,} → {rows_after:,} ({rows_dropped:,} bound violations)")
    else:
        print(f"  ✅ {df_name}: {rows_before:,} rows (all values within bounds)")

drop_percentage = (total_violations / total_rows_before) * 100 if total_rows_before > 0 else 0

print(f"\n✅ R1/R2 bounds validation completed!")
print(f"   📊 Total rows before: {total_rows_before:,}")
print(f"   📊 Total rows after: {total_rows_after:,}")
print(f"   🗑️ Total violations removed: {total_violations:,} ({drop_percentage:.2f}%)")
print(f"   📋 Dataframes processed: {len(r_dataframes)}")

# Summary of violations by r-type
if violation_summary:
    print(f"\n📋 Violations by correlation coefficient type:")
    for r_type, violations in violation_summary.items():
        print(f"   {r_type}:")
        print(f"    - Values < 0: {violations['lower']:,}")
        print(f"    - Values > 1: {violations['upper']:,}")
        print(f"    - Total: {violations['total']:,}")

# Verify data quality after bounds enforcement
print(f"\n🔍 Post-validation r1/r2 value ranges:")
if r_dataframes:
    sample_df_name = r_dataframes[0]
    sample_df = globals()[sample_df_name]
    numeric_cols = [col for col in sample_df.columns if col != 'timekey' and sample_df[col].dtype in ['int64', 'float64']]
    
    if numeric_cols:
        # Show range for first few frequency bands
        sample_cols = numeric_cols[:3]  # Show first 3 frequency bands
        for col in sample_cols:
            min_val = sample_df[col].min()
            max_val = sample_df[col].max()
            print(f"   {sample_df_name} {col}: [{min_val:.4f}, {max_val:.4f}]")
        
        if len(numeric_cols) > 3:
            print(f"   ... and {len(numeric_cols) - 3} more frequency bands")

print(f"\n🎉 All r1/r2 correlation coefficients now within valid bounds [0, 1]!")


🔒 Verifying r1 and r2 correlation coefficient bounds...
🎯 Found 42 r1/r2 dataframes to verify
📋 Expected bounds: r1, r2 ∈ [0, 1] (correlation coefficients)

🔍 Validating r1/r2 bounds...
  ✅ r1_41008_2006: 739 rows (all values within bounds)
  ✅ r1_41008_2007: 8,650 rows (all values within bounds)
  ✅ r1_41008_2008: 8,650 rows (all values within bounds)
  ✅ r1_41008_2009: 3,250 rows (all values within bounds)
  ✅ r1_41008_2013: 8,041 rows (all values within bounds)
  ✅ r1_41008_2014: 8,406 rows (all values within bounds)
  ✅ r1_41008_2015: 8,644 rows (all values within bounds)
  ✅ r1_41008_2016: 6,151 rows (all values within bounds)
  ✅ r1_41008_2017: 8,583 rows (all values within bounds)
  ✅ r1_41008_2018: 8,613 rows (all values within bounds)
  ✅ r1_41008_2019: 8,586 rows (all values within bounds)
  ✅ r1_41008_2020: 8,580 rows (all values within bounds)
  ✅ r1_41008_2021: 8,490 rows (all values within bounds)
  ✅ r1_41008_2022: 8,655 rows (all values within bounds)
  ✅ r1_41008_2023:

In [25]:
# 📊 Total Row Count for Met Data (Stations 51028 and 41008)

print("📊 Total Row Count for Met Data")
print("=" * 40)

# Target stations
target_stations = ['51028', '41008']

# Get met dataframes for target stations only
met_dataframes = [name for name in globals().keys() 
                 if name.startswith('met_') 
                 and any(station in name for station in target_stations)
                 and isinstance(globals()[name], pd.DataFrame)]

total_rows = 0
station_totals = {'51028': 0, '41008': 0}

print(f"📋 Met dataframes found: {len(met_dataframes)}")

for df_name in met_dataframes:
    df = globals()[df_name]
    rows = len(df)
    total_rows += rows
    
    # Determine station
    station = df_name.split('_')[1]
    if station in station_totals:
        station_totals[station] += rows
    
    print(f"  📁 {df_name}: {rows:,} rows")

print(f"\n📊 Station Totals:")
for station, count in station_totals.items():
    print(f"  Station {station}: {count:,} rows")

print(f"\n🎯 TOTAL ROWS (both stations): {total_rows:,}")

📊 Total Row Count for Met Data
📋 Met dataframes found: 21
  📁 met_51028_2005: 6,787 rows
  📁 met_51028_2006: 7,073 rows
  📁 met_51028_2007: 2,962 rows
  📁 met_51028_2008: 2,507 rows
  📁 met_41008_2006: 739 rows
  📁 met_41008_2007: 8,650 rows
  📁 met_41008_2008: 8,650 rows
  📁 met_41008_2009: 3,250 rows
  📁 met_41008_2013: 8,041 rows
  📁 met_41008_2014: 8,406 rows
  📁 met_41008_2015: 8,644 rows
  📁 met_41008_2016: 6,151 rows
  📁 met_41008_2017: 8,583 rows
  📁 met_41008_2018: 8,613 rows
  📁 met_41008_2019: 8,586 rows
  📁 met_41008_2020: 8,580 rows
  📁 met_41008_2021: 8,490 rows
  📁 met_41008_2022: 8,655 rows
  📁 met_41008_2023: 5,322 rows
  📁 met_41008_2024: 1,825 rows
  📁 met_41008_2025: 1,791 rows

📊 Station Totals:
  Station 51028: 19,329 rows
  Station 41008: 112,976 rows

🎯 TOTAL ROWS (both stations): 132,305


🔒 Enforcing physical bounds on meteorological data...
🎯 Found 21 met dataframes to validate
📋 Physical bounds constraints:
   WSPD: >= 0 - wind speed must be non-negative
   WVHT: >= 0 - wave height must be non-negative
   DPD: > 0 - dominant period must be positive
   APD: > 0 - average period must be positive
   WDIR: [0, 360] - wind direction valid compass range [0, 360]
   MWD: [0, 360] - mean wave direction valid compass range [0, 360]

🔍 Validating physical bounds...
  ✅ met_41008_2006: 739 rows (no violations)
  ✅ met_41008_2007: 8,650 rows (no violations)
  📁 met_41008_2008: 8,652 → 8,650 (2 dropped)
    - MWD: 2 violations
  ✅ met_41008_2009: 3,250 rows (no violations)
  ✅ met_41008_2013: 8,041 rows (no violations)
  ✅ met_41008_2014: 8,406 rows (no violations)
  ✅ met_41008_2015: 8,644 rows (no violations)
  ✅ met_41008_2016: 6,151 rows (no violations)
  ✅ met_41008_2017: 8,583 rows (no violations)
  ✅ met_41008_2018: 8,613 rows (no violations)
  ✅ met_41008_2019: 8,586 rows 

In [26]:
# 🔍 Verify DataFrame Lengths are Equal for Same Station/Year
# Ensure all file types have consistent row counts for each station-year combination

print("🔍 Verifying dataframe lengths for same station/year combinations...")

# Get target configuration from previous processing
target_stations = ['51028', '41008']
file_types = ['met', 'den', 'dir1', 'dir2', 'r1', 'r2']

# Group dataframes by station and year
station_year_groups = {}

# Find all processed dataframes and group them
for df_name in list(globals().keys()):
    if (any(df_name.startswith(f'{ft}_') for ft in file_types) and 
        any(station in df_name for station in target_stations) and
        isinstance(globals()[df_name], pd.DataFrame)):
        
        # Parse dataframe name: type_station_year
        parts = df_name.split('_')
        if len(parts) >= 3:
            file_type = parts[0]
            station = parts[1]
            year = parts[2]
            
            # Create station-year key
            station_year_key = f"{station}_{year}"
            
            if station_year_key not in station_year_groups:
                station_year_groups[station_year_key] = {}
            
            station_year_groups[station_year_key][file_type] = {
                'name': df_name,
                'length': len(globals()[df_name])
            }

print(f"🎯 Found {len(station_year_groups)} station-year combinations")

# Verify length consistency for each station-year group
consistent_groups = 0
inconsistent_groups = 0
verification_results = {}

print(f"\n🔍 Checking length consistency...")

for station_year, file_group in station_year_groups.items():
    station, year = station_year.split('_')
    
    # Get lengths for all available file types
    lengths = {}
    for file_type, info in file_group.items():
        lengths[file_type] = info['length']
    
    # Check if all lengths are the same
    unique_lengths = set(lengths.values())
    is_consistent = len(unique_lengths) == 1
    
    verification_results[station_year] = {
        'consistent': is_consistent,
        'lengths': lengths,
        'files': [info['name'] for info in file_group.values()]
    }
    
    if is_consistent:
        consistent_groups += 1
        expected_length = list(unique_lengths)[0]
        print(f"  ✅ Station {station}, Year {year}: {len(lengths)} files, {expected_length:,} rows each")
    else:
        inconsistent_groups += 1
        print(f"  ❌ Station {station}, Year {year}: INCONSISTENT LENGTHS")
        for file_type, length in lengths.items():
            print(f"    - {file_type}: {length:,} rows")

# Summary report
print(f"\n📊 Verification Summary:")
print(f"   ✅ Consistent groups: {consistent_groups}")
print(f"   ❌ Inconsistent groups: {inconsistent_groups}")
print(f"   📊 Total groups checked: {len(station_year_groups)}")

if inconsistent_groups == 0:
    print(f"\n🎉 ALL DATAFRAMES HAVE CONSISTENT LENGTHS! 🎉")
    print(f"   Ready for machine learning tasks!")
else:
    print(f"\n⚠️  INCONSISTENCIES DETECTED!")
    print(f"   Need to investigate mismatched groups before proceeding")

# Detailed breakdown by station
print(f"\n📋 Breakdown by Station:")
for station in target_stations:
    station_groups = [key for key in station_year_groups.keys() if key.startswith(station)]
    if station_groups:
        station_consistent = sum(1 for key in station_groups if verification_results[key]['consistent'])
        years = sorted([int(key.split('_')[1]) for key in station_groups])
        print(f"  Station {station}: {len(station_groups)} years ({min(years)}-{max(years)})")
        print(f"    ✅ Consistent: {station_consistent}")
        print(f"    ❌ Inconsistent: {len(station_groups) - station_consistent}")


🔍 Verifying dataframe lengths for same station/year combinations...
🎯 Found 21 station-year combinations

🔍 Checking length consistency...
  ✅ Station 51028, Year 2005: 6 files, 6,787 rows each
  ✅ Station 51028, Year 2006: 6 files, 7,073 rows each
  ✅ Station 51028, Year 2007: 6 files, 2,962 rows each
  ✅ Station 51028, Year 2008: 6 files, 2,507 rows each
  ✅ Station 41008, Year 2006: 6 files, 739 rows each
  ✅ Station 41008, Year 2007: 6 files, 8,650 rows each
  ✅ Station 41008, Year 2008: 6 files, 8,650 rows each
  ✅ Station 41008, Year 2009: 6 files, 3,250 rows each
  ✅ Station 41008, Year 2013: 6 files, 8,041 rows each
  ✅ Station 41008, Year 2014: 6 files, 8,406 rows each
  ✅ Station 41008, Year 2015: 6 files, 8,644 rows each
  ✅ Station 41008, Year 2016: 6 files, 6,151 rows each
  ✅ Station 41008, Year 2017: 6 files, 8,583 rows each
  ✅ Station 41008, Year 2018: 6 files, 8,613 rows each
  ✅ Station 41008, Year 2019: 6 files, 8,586 rows each
  ✅ Station 41008, Year 2020: 6 files,

In [27]:
# 💾 Save All Processed DataFrames to Pickle Files
# Create /data/dataframes/ directory and save all processed DataFrames

print("💾 Saving processed dataframes to pickle files...")

# Create dataframes directory
dataframes_dir = Path('../data/dataframes')
dataframes_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Created directory: {dataframes_dir}")

# Get all processed dataframes (stations 51028 and 41008, all file types)
target_stations = ['51028', '41008']
file_types = ['met', 'den', 'dir1', 'dir2', 'r1', 'r2']

# Find all dataframes that match our naming pattern
all_dataframes = []
for df_name in globals().keys():
    # Check if it's a processed dataframe (matches pattern: type_station_year)
    if (any(df_name.startswith(f'{ft}_') for ft in file_types) and 
        any(station in df_name for station in target_stations) and
        isinstance(globals()[df_name], pd.DataFrame)):
        all_dataframes.append(df_name)

all_dataframes.sort()

print(f"🎯 Found {len(all_dataframes)} dataframes to save")

# Save each dataframe
saved_count = 0
failed_count = 0
total_size_mb = 0

print(f"\n💾 Saving dataframes...")

for df_name in all_dataframes:
    try:
        df = globals()[df_name]
        
        # Create file path
        pkl_file = dataframes_dir / f"{df_name}.pkl"
        
        # Save to pickle
        df.to_pickle(pkl_file)
        
        # Get file size
        file_size_mb = pkl_file.stat().st_size / (1024 * 1024)
        total_size_mb += file_size_mb
        
        saved_count += 1
        print(f"  ✅ {df_name}.pkl ({file_size_mb:.2f} MB) - Shape: {df.shape}")
        
    except Exception as e:
        failed_count += 1
        print(f"  ❌ {df_name}: Error - {str(e)}")

print(f"\n✅ Dataframe saving completed!")
print(f"   📊 Total dataframes: {len(all_dataframes)}")
print(f"   ✅ Successfully saved: {saved_count}")
print(f"   ❌ Failed to save: {failed_count}")
print(f"   💽 Total size: {total_size_mb:.2f} MB")
print(f"   📁 Saved to: {dataframes_dir.absolute()}")

# Show breakdown by file type and station
print(f"\n📋 Breakdown by type:")
for file_type in file_types:
    type_dataframes = [df for df in all_dataframes if df.startswith(f'{file_type}_')]
    if type_dataframes:
        # Count by station
        station_counts = {}
        for station in target_stations:
            station_count = len([df for df in type_dataframes if station in df])
            if station_count > 0:
                station_counts[station] = station_count
        
        station_summary = ', '.join([f"Station {s}: {c}" for s, c in station_counts.items()])
        print(f"  {file_type}: {len(type_dataframes)} files ({station_summary})")


💾 Saving processed dataframes to pickle files...
📁 Created directory: ..\data\dataframes
🎯 Found 126 dataframes to save

💾 Saving dataframes...
  ✅ den_41008_2006.pkl (0.28 MB) - Shape: (739, 48)
  ✅ den_41008_2007.pkl (3.21 MB) - Shape: (8650, 48)
  ✅ den_41008_2008.pkl (3.21 MB) - Shape: (8650, 48)
  ✅ den_41008_2009.pkl (1.21 MB) - Shape: (3250, 48)
  ✅ den_41008_2013.pkl (2.98 MB) - Shape: (8041, 48)
  ✅ den_41008_2014.pkl (3.12 MB) - Shape: (8406, 48)
  ✅ den_41008_2015.pkl (3.21 MB) - Shape: (8644, 48)
  ✅ den_41008_2016.pkl (2.28 MB) - Shape: (6151, 48)
  ✅ den_41008_2017.pkl (3.19 MB) - Shape: (8583, 48)
  ✅ den_41008_2018.pkl (3.20 MB) - Shape: (8613, 48)
  ✅ den_41008_2019.pkl (3.19 MB) - Shape: (8586, 48)
  ✅ den_41008_2020.pkl (3.18 MB) - Shape: (8580, 48)
  ✅ den_41008_2021.pkl (3.15 MB) - Shape: (8490, 48)
  ✅ den_41008_2022.pkl (3.21 MB) - Shape: (8655, 48)
  ✅ den_41008_2023.pkl (1.98 MB) - Shape: (5322, 48)
  ✅ den_41008_2024.pkl (0.68 MB) - Shape: (1825, 48)
  ✅ den_4

In [28]:
# 📂 Load All Processed DataFrames from Pickle Files
# Load all saved dataframes from /data/dataframes/ directory

print("📂 Loading processed dataframes from pickle files...")

# Define dataframes directory path
dataframes_dir = Path('../data/dataframes')

# Check if directory exists
if not dataframes_dir.exists():
    print(f"❌ Directory not found: {dataframes_dir}")
    print("   Make sure to run the saving cell first!")
else:
    print(f"📁 Loading from: {dataframes_dir.absolute()}")
    
    # Find all pickle files in the directory
    pkl_files = list(dataframes_dir.glob("*.pkl"))
    
    if not pkl_files:
        print("❌ No pickle files found in directory!")
    else:
        print(f"🎯 Found {len(pkl_files)} pickle files to load")
        
        # Load each dataframe
        loaded_count = 0
        failed_count = 0
        total_size_mb = 0
        loaded_dataframes = []
        
        print(f"\n📂 Loading dataframes...")
        
        for pkl_file in sorted(pkl_files):
            try:
                # Extract dataframe name from filename (remove .pkl extension)
                df_name = pkl_file.stem
                
                # Load dataframe from pickle
                df = pd.read_pickle(pkl_file)
                
                # Create variable in global namespace
                globals()[df_name] = df
                
                # Get file size
                file_size_mb = pkl_file.stat().st_size / (1024 * 1024)
                total_size_mb += file_size_mb
                
                loaded_count += 1
                loaded_dataframes.append(df_name)
                
                print(f"  ✅ {df_name} ({file_size_mb:.2f} MB) - Shape: {df.shape}")
                
            except Exception as e:
                failed_count += 1
                print(f"  ❌ {pkl_file.name}: Error - {str(e)}")
        
        print(f"\n✅ Dataframe loading completed!")
        print(f"   📊 Total files found: {len(pkl_files)}")
        print(f"   ✅ Successfully loaded: {loaded_count}")
        print(f"   ❌ Failed to load: {failed_count}")
        print(f"   💽 Total size loaded: {total_size_mb:.2f} MB")
        
        # Organize loaded dataframes by type and station
        if loaded_dataframes:
            print(f"\n📋 Loaded DataFrame Summary:")
            
            # Expected file types and stations
            file_types = ['met', 'den', 'dir1', 'dir2', 'r1', 'r2']
            target_stations = ['51028', '41008']
            
            # Count by file type and station
            for file_type in file_types:
                type_dfs = [df for df in loaded_dataframes if df.startswith(f'{file_type}_')]
                if type_dfs:
                    # Count by station
                    station_counts = {}
                    for station in target_stations:
                        station_count = len([df for df in type_dfs if station in df])
                        if station_count > 0:
                            station_counts[station] = station_count
                    
                    if station_counts:
                        station_summary = ', '.join([f"Station {s}: {c}" for s, c in station_counts.items()])
                        print(f"  {file_type}: {len(type_dfs)} files ({station_summary})")
            
            # Show sample shapes for verification
            print(f"\n🔍 Sample DataFrame Info:")
            sample_dfs = loaded_dataframes[:5]  # Show first 5 as examples
            for df_name in sample_dfs:
                df = globals()[df_name]
                print(f"  {df_name}: {df.shape} - Columns: {list(df.columns)[:3]}...")
            
            if len(loaded_dataframes) > 5:
                print(f"  ... and {len(loaded_dataframes) - 5} more dataframes")
        
        print(f"\n🎉 All dataframes are now available in the global namespace!")
        print(f"   Ready for analysis and machine learning tasks!")


📂 Loading processed dataframes from pickle files...
📁 Loading from: d:\projects\FYP-OCNWVS\notebooks\..\data\dataframes
🎯 Found 126 pickle files to load

📂 Loading dataframes...
  ✅ den_41008_2006 (0.28 MB) - Shape: (739, 48)
  ✅ den_41008_2007 (3.21 MB) - Shape: (8650, 48)
  ✅ den_41008_2008 (3.21 MB) - Shape: (8650, 48)
  ✅ den_41008_2009 (1.21 MB) - Shape: (3250, 48)
  ✅ den_41008_2013 (2.98 MB) - Shape: (8041, 48)
  ✅ den_41008_2014 (3.12 MB) - Shape: (8406, 48)
  ✅ den_41008_2015 (3.21 MB) - Shape: (8644, 48)
  ✅ den_41008_2016 (2.28 MB) - Shape: (6151, 48)
  ✅ den_41008_2017 (3.19 MB) - Shape: (8583, 48)
  ✅ den_41008_2018 (3.20 MB) - Shape: (8613, 48)
  ✅ den_41008_2019 (3.19 MB) - Shape: (8586, 48)
  ✅ den_41008_2020 (3.18 MB) - Shape: (8580, 48)
  ✅ den_41008_2021 (3.15 MB) - Shape: (8490, 48)
  ✅ den_41008_2022 (3.21 MB) - Shape: (8655, 48)
  ✅ den_41008_2023 (1.98 MB) - Shape: (5322, 48)
  ✅ den_41008_2024 (0.68 MB) - Shape: (1825, 48)
  ✅ den_41008_2025 (0.67 MB) - Shape: (

In [29]:
dir2_41008_2018.head()

,timekey,.0200,.0325,.0375,.0425,.0475,.0525,.0575,.0625,.0675,...,.3300,.3400,.3500,.3650,.3850,.4050,.4250,.4450,.4650,.4850
0,2018010100,104,297,105,256,263,292,91,74,247,...,53,35,39,36,26,25,19,71,10,28
1,2018010101,275,283,114,111,100,112,114,71,86,...,43,45,36,49,29,24,17,26,360,15
2,2018010102,293,88,125,129,162,127,62,234,261,...,41,51,42,54,38,40,49,43,78,30
3,2018010103,189,344,169,162,340,332,344,222,57,...,44,32,35,20,36,24,23,35,40,36
4,2018010104,14,15,22,200,194,300,20,20,16,...,57,51,53,44,46,24,35,71,341,357
